In [8]:
# imports
import numpy as np
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict

Load trajectories

In [9]:
with open("trajectories.pickle", "rb") as f:
    trajectories = pickle.load(f)

print(type(trajectories), len(trajectories))
print("Example episode type:", type(trajectories[0]))
print("Example episode length:", len(trajectories[0]))
print("Example first step:", trajectories[0][0])



<class 'list'> 1000
Example episode type: <class 'list'>
Example episode length: 169
Example first step: ((np.int32(15), np.int32(0)), np.int64(0), np.float64(-0.5706047114070424), (np.int32(15), np.int32(1)))


v_π(s) (state-value)
“How good is state s if we keep following policy π?”
q_π(s,a) (action-value)
“How good is taking action 𝑎 in state 𝑠, and then continuing with π?”

Task 1:

MC evaluation for vπ(s)

In [10]:
def mc_eval_V(trajectories, gamma=1.0, first_visit=True):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for episode in trajectories:
        # episode is a list of (s, a, r, s_next)
        states = [step[0] for step in episode]
        rewards = [float(step[2]) for step in episode]  # r_t

        # compute returns G_t backwards
        G = 0.0
        returns = [0.0] * len(rewards)
        for t in reversed(range(len(rewards))):
            G = rewards[t] + gamma * G
            returns[t] = G

        visited = set()
        for t, s in enumerate(states):
            if first_visit and s in visited:
                continue
            visited.add(s)
            returns_sum[s] += returns[t]
            returns_count[s] += 1

    V = {s: returns_sum[s] / returns_count[s] for s in returns_sum}
    return V

In [11]:
V = mc_eval_V(trajectories, gamma=1.0, first_visit=True)
print("Number of states estimated:", len(V))
print("V((15,0)):", V.get((15,0)))

Number of states estimated: 2924
V((15,0)): -78.13585565252045


In [12]:
def mc_eval_Q(trajectories, gamma=1.0, first_visit=True):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)

    for episode in trajectories:
        states = [step[0] for step in episode]
        actions = [int(step[1]) for step in episode]
        rewards = [float(step[2]) for step in episode]

        G = 0.0
        returns = [0.0] * len(rewards)
        for t in reversed(range(len(rewards))):
            G = rewards[t] + gamma * G
            returns[t] = G

        visited = set()
        for t, (s, a) in enumerate(zip(states, actions)):
            key = (s, a)
            if first_visit and key in visited:
                continue
            visited.add(key)
            returns_sum[key] += returns[t]
            returns_count[key] += 1

    Q = {k: returns_sum[k] / returns_count[k] for k in returns_sum}
    return Q

In [13]:
Q = mc_eval_Q(trajectories, gamma=1.0, first_visit=True)
print("Number of (s,a) estimated:", len(Q))
print("Q((15,0),0):", Q.get(((15,0),0)))

Number of (s,a) estimated: 14138
Q((15,0),0): -77.43312773207876
